In [ ]:
!pip install datasets

In [ ]:
!pip install datasets==2.16.1

In [ ]:
from datasets import load_dataset
dataset = load_dataset("conll2003")

In [ ]:
print(dataset)

In [ ]:
print(dataset["train"][0])

In [ ]:
label_list = dataset["train"].features["chunk_tags"].feature.names
print(label_list)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
print(tokenized_dataset["train"][0])

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=0.1,
    weight_decay=0.01
)

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",   # ✅ ADD THIS
        max_length=128,         # ✅ ADD THIS
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
!pip install seqeval
from seqeval.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(p):
    predictions, labels = p

    predictions = predictions.argmax(axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_pred = []
        curr_lab = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_pred.append(label_list[p_val])
                curr_lab.append(label_list[l_val])

        true_predictions.append(curr_pred)
        true_labels.append(curr_lab)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

In [ ]:
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:

import torch

def predict(sentence):
    tokens = sentence.split()

    # ░ Step 1: Tokenize WITHOUT tensors
    tokenized_inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized_inputs.word_ids()

    # ░ Step 2: Convert to tensors
    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # ░ Step 3: Prediction
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)

    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        elif word_idx != previous_word_idx:
            predicted_labels.append(label_list[predictions[0][idx].item()])
        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

In [ ]:
sentence = "John works at Google in California"
print(predict(sentence))

POS Tagging vs Chunking

POS Tagging (Part-of-Speech Tagging)

POS tagging assigns grammatical labels to each individual word in a sentence. These labels include noun, verb, adjective, adverb, etc. It focuses on understanding the role of each word independently.

Example:
Sentence: John works at Google
POS Tags:

John → Noun
works → Verb
at → Preposition
Google → Proper Noun

👉 This is word-level classification, making it relatively simple and easier to model.

Chunking (Phrase Detection)

Chunking groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP). It captures relationships between words rather than treating them independently.

Example:

[John] → Noun Phrase (NP)
[works at Google] → Verb Phrase (VP)

👉 This is phrase-level classification, requiring contextual understanding.

#REPORT
Differences

POS tagging labels each word individually, while chunking groups words into phrases. Chunking requires understanding relationships between words, making it more complex than POS tagging.

🔹 Challenges Faced (IMPORTANT — evaluators look for this)

Be honest but technical:

Handling subword tokenization in BERT
Aligning labels correctly using word_ids()

Assigning -100 to ignore special tokens
Managing dataset compatibility issues (library version conflicts)

High training time due to transformer complexity

Difficulty in interpreting sequence-level evaluation metrics

👉 Don’t remove the “version issues” point — it shows real experience.

🔹 Observations and Insights
BERT performs well for sequence labeling tasks due to contextual embeddings

Proper label alignment significantly impacts model performance

Chunking gives deeper linguistic understanding compared to POS tagging

Evaluation using F1 score is more meaningful than accuracy

Pretrained models reduce the need for large datasets

Pipeline Flow

Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison